In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()


# preview 10 rows
q("""
SELECT *
FROM `prc-data-challenge-2025.prc2025.flightlist_train`
LIMIT 10
""")

We have the information about de takeoff and destination airport thanks to the ICAO, and the times associate. Also have the info about the type of plane. One row = one flight. 
Thanks to the flight_id we can link the table to the fuel table.
Thanks to the *_icao we can link the table to the airport table.
takeoff and landed are already in the date format (UTC)

In [ ]:
# global counts, period and null values
q("""
    SELECT COUNT(*) as n_flights,
    Min(takeoff) as min_takeoff,
    Max(takeoff) as max_takeoff,
    Min(landed) as min_landed,
    Max(landed) as max_landed,
    COUNT(DISTINCT flight_id) as n_unique_flights,
    COUNT(DISTINCT origin_icao) as n_unique_origin_icao,
    COUNT(DISTINCT destination_icao) as n_unique_destination_icao,
    COUNT(DISTINCT aircraft_type) as n_unique_aircraft_type,
    COUNTIF(takeoff IS NULL) as n_null_takeoff,
    COUNTIF(landed IS NULL) as n_null_landed,
    COUNTIF(flight_id IS NULL) as n_null_flight_id,
    COUNTIF(aircraft_type IS NULL) as n_null_aircraft_type
    FROM `prc-data-challenge-2025.prc2025.flightlist_train`
    
""")

11037 flights, flight_id is a unique key (11037 rows = 11037 distinct ids).
Period: first takeoff 2025-04-12, last landing 2025-09-01.
273 origin airports, 260 destination airports, 26 aircraft types.
No missing values.

In [ ]:
# flights per aircraft type
q("""
    SELECT COUNT(*) as n_flights,aircraft_type,
    FROM prc-data-challenge-2025.prc2025.flightlist_train
    GROUP BY aircraft_type
    ORDER BY n_flights DESC
""")

Flights per aircraft type: very unbalanced.
A20N + A320 = 6243 flights (57%), top 3 with A359 = 71%: mostly A320 family (short/medium haul).
13 types have fewer than 30 flights (123 flights in total), 3 types have a single flight (MD11, B39M, A318).
-> the model will have very little data for rare types: grouping them (by family or engine count) may be needed.
-> rare 4-engine types (B744, B748, A388) and cargo types (A306, MD11): candidates to explain the extreme fuel rates.

In [ ]:
# flight duration (min)
q("""
    SELECT TIMESTAMP_DIFF(landed, takeoff, MINUTE) as flight_duration,
    FROM `prc-data-challenge-2025.prc2025.flightlist_train`
    ORDER BY flight_duration DESC
    """)

The longuest flight was 1051/60= 17h, it is a longgggg flight.
The shortest one was 20 min whiwh is very very short.

In [ ]:
# median flight duration per aircraft type
q("""
    SELECT APPROX_QUANTILES(TIMESTAMP_DIFF(landed, takeoff, MINUTE), 4)[OFFSET(2)] as 
    median_flight_duration, aircraft_type,COUNT(*) as n_flights
    FROM `prc-data-challenge-2025.prc2025.flightlist_train`
    GROUP BY aircraft_type
    ORDER BY median_flight_duration DESC
    """)

Median flight duration per aircraft type (with n_flights):
- widebodies (A359, B77W, B789, B788, B772, B744): 9 to 11 h -> long haul.
- A320 family (A20N 4 h, A320 3 h 15, A21N/A321 ~2 h 45) and B737 family (~3 h): medium haul, rather long for these aircraft.
- A20N + A320 = 57% of flights -> they set the global median at ~4 h.
- rare types (A318, B39M, MD11: 1 flight; B763, B77L: 2; B748, B752: 3; A388: 4): medians not reliable.
- A306 / MD11: short flights, likely cargo.